In [53]:
import os
import pandas as pd
from pprint import pprint
import json

In [54]:
os.getcwd()

'c:\\Users\\Lukeg\\Desktop\\Capstone Virtual Environment\\Capstone_Project_AIM\\LeafletJS\\JSON'

In [55]:
from pathlib import Path

# Simple: csv files folder is in the same directory as this notebook
csv_folder = Path("csv files")

# List all CSV files
csv_files = sorted([f.name for f in csv_folder.glob('*.csv')])

print(f"📄 Found {len(csv_files)} CSV files: {csv_files}")

📄 Found 5 CSV files: ['A1.csv', 'B1.csv', 'H1.csv', 'M1.csv', 'M2.csv']


In [56]:
print(csv_files)

['A1.csv', 'B1.csv', 'H1.csv', 'M1.csv', 'M2.csv']


In [57]:
building_names = {
    "A": "Building A",
    "B": "Building B",
    "C": "Building C",
    "D": "Building D",
    "E": "Building E",
    "F": "F Building",
    "G": "Building G",
    "H": "Building H",
    "J": "Building J",
    "K": "Building K",
    "M": "Building M",
    "T": "Building T",
}

In [58]:
all_docs = {}
for file in csv_files:
    # Read from the csv_folder path
    file_path = csv_folder / file
    doc = pd.read_csv(file_path)
    
    name = file.replace(".csv", "")
    
    building_code = name[0]
    floor_number = name[1:]
    
    building_name = building_names[building_code]
    if building_name not in all_docs:
        all_docs[building_name] = {}
    all_docs[building_name][floor_number] = doc

print(f"✅ Loaded {len(all_docs)} buildings")

✅ Loaded 4 buildings


In [59]:
object_types = ["room", "building_connection", "stairs", "elevator", "outside_exit", "bathroom", "eatery"]
connects_floorplans = ["stairs", "elevator", "building_connection"]

In [60]:
accepted_node_types = ["object", "turn", "intersection", "roundabout"]

In [61]:
def object_logic(row, rooms, exits):
    object_type = row['Object_category_type']
    if not isinstance(object_type, str):
        return rooms, exits
    object_type = object_type.lower().replace(" ", "")
    object_type_list = object_type.split(",")

    objects = row['Object']
    objects = objects.replace(" ", "")
    object_list = objects.split(",")

    doors = row['Door']
    if isinstance(doors, str):
        doors = doors.replace(" ", "")
        door_list = doors.split(",")
        
    
    exit_conns = row['Exit_Connections']
    exit_conns_list = []
    if isinstance(exit_conns, str):
        exit_conns = exit_conns.replace(" ", "")
        exit_conns_raw = exit_conns.split(",")
        for element in exit_conns_raw:
            clean = element.strip("[]")
            exit_entries = clean.split("|")
            exit_conns_list.append(exit_entries)
            
    
    door_counter = 0
    exits_counter = 0
    for i,entry in enumerate(object_type_list):
        selected_obj = object_list[i]
        if entry == "room":
            door = door_list[door_counter]

            if selected_obj not in rooms:
                rooms[selected_obj] = []
            rooms[selected_obj].append(door)
            door_counter += 1
        elif entry == "exit":
            # print("exit")
            # print(object_type_list)
            # print(exit_conns_list)
            exit_destination = exit_conns_list[exits_counter]
            if selected_obj not in exits:
                exits[selected_obj] = []
            exits[selected_obj].extend(exit_destination)
            exits_counter += 1

    return rooms, exits

In [ ]:
def split_csv(value):
    if isinstance(value, str):
        return value.replace(" ", "").split(",")
    return []


def parse_exit_conns(value):
    if not isinstance(value, str):
        return []
    cleaned = value.replace(" ", "")
    return [item.strip("[]").split("|") for item in cleaned.split(",")]


def node_navigation(row, navigationGraph, roomToNode, exitToNode):

    node = row["Node"]
    connections = split_csv(row["Node_connections"])
    object_ids = split_csv(row["Object"])
    node_types = split_csv(str(row["Node_type"]).lower())
    object_types = split_csv(str(row["Object_type"]).lower())
    categories = split_csv(str(row["Object_category_type"]).lower())
    doors = split_csv(row["Door"])
    exit_conns = parse_exit_conns(row["Exit_Connections"])

    if node not in navigationGraph:
        navigationGraph[node] = {"connections": connections, "represents": []}

    represents = navigationGraph[node]["represents"]

    # Map short names → proper names
    SIMPLE_TYPES = {
        "int": "intersection",
        "turn": "turn",
        "roundabout": "roundabout"
    }

    # ------------------------------------------------------------
    # 1. Add simple node types FIRST (turn, intersection, etc.)
    # ------------------------------------------------------------
    has_only_simple = True
    for nt in node_types:
        if nt in SIMPLE_TYPES:
            represents.append({"type": SIMPLE_TYPES[nt]})
        else:
            has_only_simple = False

    # If node is ONLY a simple type with no objects, stop here
    if has_only_simple and not object_types:
        return navigationGraph, roomToNode, exitToNode

    # ------------------------------------------------------------
    # 2. Now add objects (room, stairs, exit, bathroom, etc.)
    # ------------------------------------------------------------
    door_idx = 0
    exit_idx = 0

    for obj_type, obj_id, cat in zip(object_types, object_ids, categories):

        rep = {"type": obj_type, "id": obj_id}

        if cat == "room":
            if door_idx < len(doors):
                rep["door"] = doors[door_idx]
                door_idx += 1
            
            # 🔧 UPDATED: Store rooms as arrays for multi-node support
            if obj_id not in roomToNode:
                roomToNode[obj_id] = []
            roomToNode[obj_id].append(node)

        elif cat == "exit":
            if exit_idx < len(exit_conns):
                rep["goesTo"] = exit_conns[exit_idx]
                exit_idx += 1
            else:
                rep["goesTo"] = None
            
            # 🔧 UPDATED: Store exits as arrays for multi-node support
            if obj_id not in exitToNode:
                exitToNode[obj_id] = []
            exitToNode[obj_id].append(node)

        represents.append(rep)

    return navigationGraph, roomToNode, exitToNode


In [63]:
#print(doc.columns)
def floorplan_json_generator(doc):
    navigationGraph = {}
    roomToNode = {}
    entranceToNode = {}
    objects = {
        'rooms' : None,
        'exits' : None
              }
    rooms = {}
    exits = {}
    for index, row in doc.iterrows():

        rooms, exits = object_logic(row, rooms, exits)
        navigationGraph, roomToNode, exitToNode = node_navigation(row, navigationGraph, roomToNode, entranceToNode)


    # #Sorting to make consistent
    # navigationGraph = dict(sorted(
    #     navigationGraph.items(),
    #     key=lambda x: int(x[0].split("_")[1])
    #     if "_" in x[0] and x[0].split("_")[1].isdigit()
    #     else float('inf')
    # ))
    
    objects['rooms'] = rooms
    objects['exits'] = exits
    return navigationGraph, roomToNode, entranceToNode, objects
    

In [64]:
import json

node_data = {}
for building, data in all_docs.items():
    building_code = building[-1]
    building_path = f"Floorplans/{building}"
    for floor in data:
        floorplan_code = f"{building_code}{floor}"
        svg_file = f"{floorplan_code}.svg"
        doc = data[floor]
        nav_graph, room_to_node, exit_to_node, objects = floorplan_json_generator(doc)
        if building not in node_data:
            node_data[building] = {"path": building_path,
                                   "floors" : {}
                                  }
        node_data[building]["floors"][floor] = {"plan": svg_file,
                                      "navigationGraph" : nav_graph,
                                      "roomToNode": room_to_node,
                                      "exitToNode": exit_to_node,
                                      "objects": objects}

# Output to current directory (JSON folder)
with open("all_node_data.json", "w") as f:
    json.dump(node_data, f, indent=4)

print(f"✅ Successfully wrote all_node_data.json")

✅ Successfully wrote all_node_data.json


In [65]:
pprint(node_data)

{'Building A': {'floors': {'1': {'exitToNode': {},
                                 'navigationGraph': {},
                                 'objects': {'exits': {}, 'rooms': {}},
                                 'plan': 'A1.svg',
                                 'roomToNode': {}}},
                'path': 'Floorplans/Building A'},
 'Building B': {'floors': {'1': {'exitToNode': {},
                                 'navigationGraph': {},
                                 'objects': {'exits': {}, 'rooms': {}},
                                 'plan': 'B1.svg',
                                 'roomToNode': {}}},
                'path': 'Floorplans/Building B'},
 'Building H': {'floors': {'1': {'exitToNode': {'Elevator_H1': 'H1_3',
                                                'F-Building_H1': 'H1_17',
                                                'M-Building_H1': 'M1_entry',
                                                'Outside-Exit_1_H1': 'H1_4',
                                   